In [1]:
%%capture install_log
# !pip install -q ultralytics pydicom opencv-python-headless tqdm pandas openpyxl scikit-learn pyyaml

In [2]:
# !pip install monai

In [3]:
import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'  # max_pool3d gibi MPS'de olmayan op'lar CPU'ya düşer

import glob
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import monai
from monai.transforms import (
    Compose,
    EnsureChannelFirstd,
    ScaleIntensityRanged,
    Resized,
    ToTensord,
    RandRotate90d,
    RandFlipd,
    RandGaussianNoised,
    RandZoomd
)
from sklearn.model_selection import GroupKFold
from tqdm import tqdm


/Users/ramazanpolat/Desktop/datasets/abdomen/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Cihaz Kontrolü — M5 Pro için MPS öncelikli
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f"Aktif Cihaz: {device}")

# Kaggle Dizin Yolları
DATA_DIR = "/Volumes/Yeni Birim/rsna-2023-abdominal-trauma-detection"
TRAIN_IMAGES_DIR = os.path.join(DATA_DIR, "train_images")
TRAIN_CSV = os.path.join(DATA_DIR, "train_2024.csv")


Aktif Cihaz: mps


In [5]:
from concurrent.futures import ProcessPoolExecutor, as_completed, TimeoutError
from model_architecture import _read_and_hu

CACHE_DIR = 'kaggle/working/npy_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

def preprocess_all(patient_ids, train_images_dir, cache_dir,
                   num_workers=6):
    args_list = [(pid, train_images_dir, cache_dir) for pid in patient_ids]
    missing = []

    with ProcessPoolExecutor(max_workers=num_workers) as pool:
        futures = {pool.submit(_read_and_hu, a): a[0] for a in args_list}
        for future in tqdm(as_completed(futures), total=len(futures), desc='DICOM okuma'):
            pid = futures[future]
            try:
                pid, volume = future.result(timeout=120)
            except TimeoutError:
                print(f'\n[TIMEOUT] hasta {pid} atlanıyor')
                missing.append(pid)
                future.cancel()
                continue
            except Exception as e:
                print(f'\n[HATA] hasta {pid}: {e}')
                missing.append(pid)
                continue
            if volume is None:
                continue  # zaten cache'de var
            np.save(os.path.join(cache_dir, f'{pid}.npy'), volume)

    print(f'Tamamlandi. Eksik/hata: {len(missing)} hasta')
    return missing

# Calistir
df_all      = pd.read_csv(TRAIN_CSV)
patient_ids = df_all['patient_id'].unique()
missing     = preprocess_all(patient_ids, TRAIN_IMAGES_DIR, CACHE_DIR)


DICOM okuma: 100%|██████████| 3147/3147 [00:03<00:00, 867.97it/s] 


Tamamlandi. Eksik/hata: 0 hasta


In [ ]:
from model_architecture import FastRSNADataset, RSNAResNetCBAMClassifier


df_train = pd.read_csv(TRAIN_CSV)

gkf = GroupKFold(n_splits=5)
splits = list(gkf.split(df_train, groups=df_train['patient_id']))
train_idx, val_idx = splits[0]
df_tr, df_va = df_train.iloc[train_idx], df_train.iloc[val_idx]

# FastRSNADataset: .npy okur, DICOM degil
train_ds = FastRSNADataset(df_tr, CACHE_DIR, augment=True)
val_ds   = FastRSNADataset(df_va, CACHE_DIR, augment=False)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,
                          num_workers=4, pin_memory=True,
                          persistent_workers=True, prefetch_factor=2)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False,
                          num_workers=4, pin_memory=True,
                          persistent_workers=True, prefetch_factor=2)

# Model
model = RSNAResNetCBAMClassifier(num_classes=13)
if torch.cuda.device_count() > 1:
    print(f'DataParallel: {torch.cuda.device_count()} GPU aktif')
    model = nn.DataParallel(model)
model = model.to(device)

# Loss, Optimizer, Scheduler
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-2)
num_epochs = 50
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=2e-4,
    steps_per_epoch=len(train_loader),
    epochs=num_epochs, pct_start=0.1
)
scaler = GradScaler(device=device.type)

# Egitim Dongusu
best_val_loss = float('inf')
print('\n--- RSNA 2023 Egitimi Basliyor ---')

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs} Train'):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type=device.type):
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running_loss += loss.item() * images.size(0)
    train_loss = running_loss / len(train_loader.dataset)

    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f'Epoch {epoch+1}/{num_epochs} Val'):
            images, labels = images.to(device), labels.to(device)
            with autocast(device_type=device.type):
                outputs = model(images)
                loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
    val_loss = running_loss / len(val_loader.dataset)

    lr_now = scheduler.get_last_lr()[0]
    print(f'Epoch {epoch+1:02d}/{num_epochs} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | LR: {lr_now:.2e}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
        torch.save(state, '/kaggle/working/rsna2023_best_attention_model.pth')
        print('  => Best model kaydedildi')


[FastRSNADataset] 771 hasta cache bulunamadi, atlandi.
[FastRSNADataset] 196 hasta cache bulunamadi, atlandi.
Train: 1746 | Val: 434

--- RSNA 2023 Egitimi Basliyor ---


Epoch 1/50 Train:   0%|          | 0/219 [00:00<?, ?it/s]/Users/ramazanpolat/Desktop/datasets/abdomen/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/ramazanpolat/Desktop/datasets/abdomen/.venv/lib/python3.9/site-packages/torch/nn/functional.py:917: UserWarning: The operator 'aten::max_pool3d_with_indices' is not currently supported on the MPS backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/mps/MPSFallback.mm:15.)
  return torch.max_pool3d(input, kernel_size, stride, padding, dilation, ceil_mode)
Epoch 1/50 Train:  38%|███▊      | 84/219 [1:36:35<2:41:16, 71.68s/it]

In [ ]:
# Cihaz Kontrolü
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Aktif Cihaz: {device}")

# T4 x2: kaç GPU var?
n_gpu = torch.cuda.device_count()
print(f"GPU sayısı: {n_gpu}")

# Kaggle Dizin Yolları
DATA_DIR = "/kaggle/input/competitions/rsna-2023-abdominal-trauma-detection"
TRAIN_IMAGES_DIR = os.path.join(DATA_DIR, "train_images")
TRAIN_CSV = os.path.join(DATA_DIR, "train_2024.csv")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [ ]:
# --- FAZ 1 & FAZ 2: ÖN İŞLEME VE VERİ ARTIRIMI (AUGMENTATION) ---

# Train Transform: Ön işleme + Faz 2 Regülasyonları
train_transforms = Compose([
    EnsureChannelFirstd(keys=["image"], channel_dim="no_channel"),
    # Faz 1: HU Clipping [-150, 250] ve [0, 1] normalizasyonu
    ScaleIntensityRanged(keys=["image"], a_min=-150, a_max=250, b_min=0.0, b_max=1.0, clip=True),
    # Faz 1: 128x128x128 Hacimsel Yeniden Boyutlandırma
    Resized(keys=["image"], spatial_size=(128, 128, 128)),
    
    # Faz 2: Farklı tarama oryantasyonları ve gürültüler için Augmentation adımları
    RandRotate90d(keys=["image"], prob=0.5, spatial_axes=(0, 1)),
    RandFlipd(keys=["image"], prob=0.5, spatial_axis=0),
    RandZoomd(keys=["image"], prob=0.3, min_zoom=0.9, max_zoom=1.1, mode="trilinear"),
    RandGaussianNoised(keys=["image"], prob=0.2, mean=0.0, std=0.05),
    
    ToTensord(keys=["image"])
])

# Validation Transform: Sadece Ön İşleme (Augmentation uygulanmaz)
val_transforms = Compose([
    EnsureChannelFirstd(keys=["image"], channel_dim="no_channel"),
    ScaleIntensityRanged(keys=["image"], a_min=-150, a_max=250, b_min=0.0, b_max=1.0, clip=True),
    Resized(keys=["image"], spatial_size=(128, 128, 128)),
    ToTensord(keys=["image"])
])

Dinamik Derinlik: Slice sayılarındaki aşırı dalgalanma yüzünden Faz 1'de uyguladığımız Resized(spatial_size=(128, 128, 128)) adımı hayat kurtarıcıdır. Az slice'lı hastaları doldurur, çok slice'lı hastaları sıkıştırarak modeli standart bir boyuta zorlar.


In [ ]:

# Modeli tekrar ana döngüye bağlayalım
from model_architecture import RSNAResNetCBAMClassifier


model = RSNAResNetCBAMClassifier(num_classes=13).to(device)
print("Süper! İç değişken bağımlılıkları tamamen ezildi ve model eğitime hazır hale getirildi. 🚀")

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(loader, desc="Eğitim Adımı"):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        # Mixed Precision (FP16) ile GPU Hızlandırma
        with autocast(device_type=device.type):
            outputs = model(images)
            loss = criterion(outputs, labels)
            
        # 1. Loss değerini ölçekle ve geriye yayılımı yap (Doğru)
        scaler.scale(loss).backward()
        
        # 2. Hatalı olan scaler.scale(optimizer).step() yerine DOĞRU kullanım:
        scaler.step(optimizer)
        
        # 3. Scaler faktörünü bir sonraki adım için güncelle
        scaler.update()
        
        running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)

In [ ]:
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Doğrulama Adımı"):
            images, labels = images.to(device), labels.to(device)
            
            # Hatalı olan boş autocast() yerine cihaz tipini belirtiyoruz:
            with autocast(device_type=device.type):
                outputs = model(images)
                loss = criterion(outputs, labels)
                
            running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)

In [ ]:
from model_architecture import RSNA20233DDataset

# ── Veri ──────────────────────────────────────────────────────────────────
df_train = pd.read_csv(TRAIN_CSV)

gkf = GroupKFold(n_splits=5)
splits = list(gkf.split(df_train, groups=df_train["patient_id"]))
train_idx, val_idx = splits[0]
df_tr, df_va = df_train.iloc[train_idx], df_train.iloc[val_idx]

train_ds = RSNA20233DDataset(df_tr, TRAIN_IMAGES_DIR, transform=train_transforms)
val_ds   = RSNA20233DDataset(df_va, TRAIN_IMAGES_DIR, transform=val_transforms)

# num_workers=4 (Kaggle 4 core), persistent_workers epoch arası restart önler
# prefetch_factor=2 GPU boşta kalmasın diye sonraki batch'i önceden hazırlar
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,
                          num_workers=4, pin_memory=True,
                          persistent_workers=True, prefetch_factor=2)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False,
                          num_workers=4, pin_memory=True,
                          persistent_workers=True, prefetch_factor=2)

# ── Model — DataParallel ile T4 x2 ───────────────────────────────────────
model = RSNAResNetCBAMClassifier(num_classes=13)
if torch.cuda.device_count() > 1:
    print(f"DataParallel: {torch.cuda.device_count()} GPU aktif")
    model = nn.DataParallel(model)
model = model.to(device)

# ── Loss, Optimizer, Scheduler ───────────────────────────────────────────
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-2)
num_epochs = 50
# OneCycleLR: warmup + cosine decay, sabit LR'den ~2× hızlı yakınsama
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=2e-4,
    steps_per_epoch=len(train_loader),
    epochs=num_epochs, pct_start=0.1
)
scaler = GradScaler(device=device.type)

# ── Eğitim Döngüsü ────────────────────────────────────────────────────────
best_val_loss = float("inf")
print("--- RSNA 2023 Eğitimi Başlıyor ---")

for epoch in range(num_epochs):
    # --- Train ---
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} Train"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type=device.type):
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running_loss += loss.item() * images.size(0)
    train_loss = running_loss / len(train_loader.dataset)

    # --- Validation ---
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} Val"):
            images, labels = images.to(device), labels.to(device)
            with autocast(device_type=device.type):
                outputs = model(images)
                loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
    val_loss = running_loss / len(val_loader.dataset)

    lr_now = scheduler.get_last_lr()[0]
    print(f"Epoch {epoch+1:02d}/{num_epochs} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | LR: {lr_now:.2e}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        # DataParallel sarmalıysa .module ile gerçek model state'i alınır
        state = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
        torch.save(state, "/kaggle/working/rsna2023_best_attention_model.pth")
        print("  => Best model kaydedildi")
